# Epidemic on a Grid (Mesa 2.x)

A minimal susceptible-infected model on a toroidal grid. People walk randomly
across a `MultiGrid`; when an infected person shares a cell with a susceptible
one, the infection spreads. Each agent carries a state, either `"S"` or `"I"`.
The notebook targets the Mesa 2.x API, including the classic `ModularServer`
browser visualisation.

In [ ]:
#!pip install mesa==2.2.0

In [ ]:
import mesa
from mesa.time import RandomActivationByType
from mesa.space import MultiGrid

In [ ]:
class Person(mesa.Agent):
    """A person in one of two epidemic states: "S" or "I"."""

    def __init__(self, unique_id, model, state):
        super().__init__(unique_id, model)
        self.state = state

    def move(self):
        neighbors = self.model.grid.get_neighborhood(
            self.pos, moore=True, include_center=False
        )
        new_pos = self.random.choice(neighbors)
        self.model.grid.move_agent(self, new_pos)

    def step(self):
        cellmates = self.model.grid.get_cell_list_contents([self.pos])
        if self.state == "I":
            for other in cellmates:
                if other.state == "S":
                    other.state = "I"
        self.move()

In [ ]:
class EpidemicModel(mesa.Model):
    """Agents move on a MultiGrid and pass an infection to their cell-mates."""

    def __init__(self, width, height, N):
        super().__init__()
        self.num_agents = N
        self.grid = MultiGrid(width, height, torus=True)
        self.schedule = RandomActivationByType(self)
        for i in range(self.num_agents):
            state = "I" if i == 0 else "S"
            a = Person(self.next_id(), self, state)
            self.schedule.add(a)
            x = self.random.randrange(width)
            y = self.random.randrange(height)
            self.grid.place_agent(a, (x, y))

    def count_occupied(self):
        occupied = 0
        for cell_content, (x, y) in self.grid.coord_iter():
            if len(cell_content) > 0:
                occupied += 1
        return occupied

    def step(self):
        self.schedule.step()

## Visualization

The block below wires up the classic Mesa 2.x browser visualisation. A
`CanvasGrid` draws the agents and a `ModularServer` serves the interactive page.
The call to `server.launch()` is left commented so the notebook does not block on
a running web server.

In [ ]:
from mesa.visualization.modules import CanvasGrid, ChartModule
from mesa.visualization.ModularVisualization import ModularServer


def agent_portrayal(agent):
    return {"Shape": "circle", "Filled": "true", "Layer": 0, "Color": "red", "r": 0.5}


grid = CanvasGrid(agent_portrayal, 10, 10, 500, 500)
server = ModularServer(
    EpidemicModel, [grid], "Epidemic", {"width": 10, "height": 10, "N": 20}
)
# server.launch()  # uncomment to run

In [ ]:
model = EpidemicModel(10, 10, 20)
for _ in range(10):
    model.step()

print("Occupied cells:", model.count_occupied())